## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-06-19-54-12 +0000,nyt,Here’s the latest.,https://www.nytimes.com/live/2026/11/06/us/mid...
1,2026-11-06-19-54-00 +0000,wsj,Saudi National Accused of Bugging OpenAI’s Off...,https://www.wsj.com/us-news/saudi-national-acc...
2,2026-11-06-19-53-55 +0000,nyt,Colom Leads Hyde-Smith in Mississippi Senate R...,https://www.nytimes.com/2026/11/06/us/election...
3,2026-11-06-19-53-18 +0000,startribune,Live: Brock Faber opens up about skull fractur...,https://www.startribune.com/minnesota-wild-san...
4,2026-11-06-19-52-37 +0000,nypost,Pete Hegseth becomes GOP’s political scapegoat...,https://nypost.com/2026/11/06/us-news/pete-heg...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2695/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
44,trump,48
102,new,26
36,midterm,22
110,market,20
105,ai,16


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
15,2026-11-06-19-37-00 +0000,wsj,Trump said that longtime adviser Stephen Mille...,https://www.wsj.com/politics/elections/trump-n...,159
5,2026-11-06-19-51-00 +0000,wsj,"Stephen Miller, who has been one of Trump’s mo...",https://www.wsj.com/politics/elections/in-tapp...,149
35,2026-11-06-18-57-40 +0000,nypost,"Trump fires Susie Wiles after midterm blowout,...",https://nypost.com/2026/11/06/us-news/trump-fi...,149
13,2026-11-06-19-42-09 +0000,startribune,Trump replaces Susie Wiles with Stephen Miller...,https://www.startribune.com/trump-replaces-sus...,131
307,2026-11-06-03-11-54 +0000,nypost,Trump pins US stock market plunge on Democrats...,https://nypost.com/2026/11/05/us-news/trump-pi...,124


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
15,159,2026-11-06-19-37-00 +0000,wsj,Trump said that longtime adviser Stephen Mille...,https://www.wsj.com/politics/elections/trump-n...
244,82,2026-11-06-11-00-00 +0000,wapo,Texas’s new Democratic governor Hinojosa vows ...,https://www.washingtonpost.com/nation/2026/11/...
430,77,2026-11-05-21-10-00 +0000,cbc,California Gov. Gavin Newsom says he'll commut...,https://www.cbc.ca/news/world/california-newso...
307,76,2026-11-06-03-11-54 +0000,nypost,Trump pins US stock market plunge on Democrats...,https://nypost.com/2026/11/05/us-news/trump-pi...
157,70,2026-11-06-15-22-00 +0000,wsj,The solid October jobs report leaves inflation...,https://www.wsj.com/economy/central-banking/so...
148,66,2026-11-06-15-32-25 +0000,chitri,Father of Meghan Markle arrested on charges re...,https://www.chicagotribune.com/2026/11/06/megh...
392,65,2026-11-05-22-26-00 +0000,wsj,The Trump administration warned conservative g...,https://www.wsj.com/politics/policy/trump-admi...
409,62,2026-11-05-21-41-00 +0000,wsj,Wall Street Gets $2.5 Trillion AI Reality Chec...,https://www.wsj.com/finance/stocks/wall-street...
380,55,2026-11-05-22-48-00 +0000,wsj,The president signed a proclamation that he sa...,https://www.wsj.com/politics/policy/trump-sign...
297,46,2026-11-06-05-32-58 +0000,bbc,Australian woman shot during Bondi Beach attac...,https://www.bbc.co.uk/news/articles/c65ynn9x50...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
